In [15]:
!pip install torchvision

In [16]:
import torch
import os
from torch.utils.data import DataLoader , Dataset
from torchvision import transforms
from PIL import Image

In [17]:
# images load =. transform => dataset of all images

class ImageProcessor:
    def __init__(self,root_dir_path,transformations = None):
        self.root_dir_path = root_dir_path
        self.transformations = transformations


        # list of paths for all images
        self.all_img_paths =  [os.path.join(root_dir_path,img)for img in os.listdir(root_dir_path)]

    def __len__(self):
        return len(self.all_img_paths)

    def __getitem__(self, idx):
        img_path = self.all_img_paths[idx] # as all_img path is a list we are acessing through index
        img = Image.open(img_path).convert("RGB")

        if self.transformations:
            img = self.transformations(img)

        return img



In [18]:

root_dir_path = "./img_align_celeba"

transformations = transforms.Compose([
    transforms.CenterCrop(178), # 178x218 => 178x178
    transforms.Resize(64),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5) ,(0.5,0.5,0.5)) # [-1,1]

])

In [19]:
dataset = ImageProcessor(root_dir_path , transformations)
print(f"loaded {len(dataset)} images")

loaded 202599 images


In [20]:
dataLoader = DataLoader(dataset,batch_size=128,shuffle = True)

Generator network

In [21]:
import torch.nn as nn
import torch.optim as optim 
import numpy as np  


In [22]:
class Generator(nn.Module):
    def __init__(self,z_dim =100,img_channels =3): # 3 is for rgb
        super(Generator,self).__init__()

        # fully connected (dense) layers
        self.model = nn.Sequential(
            nn.Linear(z_dim ,256), # 100=> 256
            nn.ReLU(),

            nn.Linear(256,512),
            nn.ReLU(),

            nn.Linear(512,1024),
            nn.ReLU(),

            nn.Linear(1024,64 * 64 * img_channels),
            nn.Tanh() # [-1,1]
        )

    def forward(self,z):
        img = self.model(z)
        img = img.view(img.size(0) ,3,64,64) # separating comining output in to image dimension (3x64x64)
        # img.size() => batch_size
        return img

    # fake image = 3x64x64xbatch => here we are fedding in batch


Discriminator work

In [23]:
class Discriminator(nn.Module):
    def __init__(self,img_channels = 3): # 3 for rgb channels
       super(Discriminator,self).__init__()

    # fully connected (dense) layer
       self.model = nn.Sequential(
            nn.Flatten(), # converting 4d to 1d as discriminator cant delas with 1d

            nn.Linear(img_channels *64*64,1024), # input channel
            nn.LeakyReLU(0.3,inplace = True),

            nn.Linear(1024,512),
            nn.LeakyReLU(0.2,inplace = True),

            nn.Linear(512,256),
            nn.LeakyReLU(0.2,inplace =True),

            nn.Linear(256,1),
            nn.Sigmoid() # PROBABILITY OF BEING REAL/FAKE

)


    def forward(self,img):
        return self.model(img)

In [24]:
GAN_loss = nn.BCELoss() # binary cross entropy

generator = Generator()
g_optimizer = optim.Adam(generator.parameters(),lr =0.0002,betas =(0.5,0.999))

discriminator = Discriminator()
d_optimizer = optim.Adam(discriminator.parameters(),lr =0.0003,betas = (0.5,0.999))

In [25]:
import torch
# device
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")

else:
    device = torch.device("cpu")

print(f"device is {device}")

device is cpu


Trainning the Gan

In [26]:
def train(generator,discriminator,dataloader,epochs =10):
    for epoch in range(epochs):
        for i,img in enumerate(dataloader):
            real_imgs  = imgs.to(device)
            batch_size = real_i.size(0)

            # create real image labels and fake imgs labels
            real_labels = torch.ones(batch_size,1).to(device) # [1,1,1,..]
            fake_labels = torch.zeros(batch_size,1).to(device) # [0,0,0,0..]

            # Train the Discriminator
            d_optimizer.zero_grad()

            fake_imgs = generator(torch.randn(batch_size,100).to(device)) # generatig random noisy_value

            real_loss = GAN_loss(discriminator(real_imgs),real_labels)
            fake_loss = GAN_loss(discriminator(fake_imgs.detach()),real_labels) # use fake_imgs but shouldn't send it back to generator through propagation

            d_loss = (real_loss+fake_loss) / 2

            d_loss.backward()
            d_optimizer.step() # to update the parameters

            # Train the generator

            g_optimizer.zero_grad()

            g_loss = GAN_loss(discriminator(fake_imgs),real_labels)

            g_loss.backward()
            g_optimizer.step()

            if i % 50 == 0:
                print(f"for epoch : {epoch+1}/{epochs}..batch :{i+1}..g_loss:{g_loss}...D_loss:{d_loss}")
        # save generated images for epoch
        save_generated_images(generator,epoch,device)
        


In [27]:
import matplotlib.pyplot as plt 
import torchvision

def save_generated_images(generator,epoch,device,num_imgs =8):
    z = torch.randn(num_imgs,100).to(device)
    generated_imgs = generator(z).detach().cpu()

    grid = torchvision.utils.make_grid(generated_imgs,nrow =4,normalize =True) # here the image generated iamges are in the form of [-1,1] but we want [0,1] form so we set normalize  =  True 
    # as here we are dealing with 2d iamges and grid
    plt.imshow(np.transpose(grid,(1,2,0)))
    plt.title(f"epoch {epoch+1}")
    plt.axis("off")
    plt.show()

In [ ]:
train(generator,discriminator,Dataloader,epochs=5)